<a href="https://colab.research.google.com/github/AayuCoder-coding/PhonepeProject/blob/main/Phonepe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Clone full dataset (Rounak36 repo includes full data)

In [21]:
!rm -rf pulse
!rm -rf PhonePe-Transaction-Insights
!git clone https://github.com/Rounak36/PhonePe-Transaction-Insights.git
!git clone https://github.com/PhonePe/pulse.git
!ls PhonePe-Transaction-Insights

Cloning into 'PhonePe-Transaction-Insights'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 12 (delta 2), reused 7 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 513.81 KiB | 1.97 MiB/s, done.
Resolving deltas: 100% (2/2), done.
Cloning into 'pulse'...
remote: Enumerating objects: 17904, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 17904 (delta 19), reused 17 (delta 17), pack-reused 17855 (from 2)
Receiving objects: 100% (17904/17904), 26.13 MiB | 11.76 MiB/s, done.
Resolving deltas: 100% (8723/8723), done.
Updating files: 100% (9029/9029), done.
aggregated_transactions.csv		README.md
aggregated_user.csv			requirements.txt.txt
phonepe_data.db				top_district_users.csv
PhonePe_Transaction_Insights_EDA.ipynb


Import required libraries

In [22]:
import os
import json
import pandas as pd
import sqlite3
import plotly.express as px
from google.colab import files

Extract National-Level Aggregated Transactions

In [23]:
def extract_aggregated_transaction_flat(path='pulse/data/aggregated/transaction/country/india/'):
    records = []
    for year in os.listdir(path):
        year_path = os.path.join(path, year)
        if not os.path.isdir(year_path): continue
        for file in os.listdir(year_path):
            file_path = os.path.join(year_path, file)
            if file.endswith('.json'):
                with open(file_path, 'r') as f:
                    data = json.load(f)
                    quarter = int(file.strip('.json'))
                    if data.get('data') and data['data'].get('transactionData'):
                        for item in data['data']['transactionData']:
                            for pi in item['paymentInstruments']:
                                records.append({
                                    'year': int(year),
                                    'quarter': quarter,
                                    'txn_type': item['name'],
                                    'count': pi['count'],
                                    'amount': pi['amount']
                                })
    print(f"✅ Extracted {len(records)} national records")
    return pd.DataFrame(records)

agg_tx_df = extract_aggregated_transaction_flat()
agg_tx_df.head()

✅ Extracted 140 national records


,year,quarter,txn_type,count,amount
0,2021,3,Merchant payments,2316924871,1.426966e+12
1,2021,3,Peer-to-peer payments,2215575286,7.429358e+12
2,2021,3,Recharge & bill payments,714583744,3.492346e+11
3,2021,3,Financial Services,3112160,4.043343e+09
4,2021,3,Others,13558464,7.146869e+09


Store in SQLite + Visualize

In [24]:
conn = sqlite3.connect("phonepe_data.db")
agg_tx_df.to_sql("aggregated_transaction", conn, if_exists='replace', index=False)
print("✅ Inserted into SQLite")

# Summary Charts
top_txn = agg_tx_df.groupby("txn_type")['amount'].sum().reset_index().sort_values(by='amount', ascending=False)
fig1 = px.bar(top_txn, x='txn_type', y='amount', title='Top Transaction Types by Total Amount')
fig1.show()

trend = agg_tx_df.groupby(['year', 'quarter'])['amount'].sum().reset_index()
trend['period'] = trend['year'].astype(str) + "-Q" + trend['quarter'].astype(str)
fig2 = px.line(trend, x='period', y='amount', title='Transaction Amount Over Time', markers=True)
fig2.show()

fig3 = px.pie(top_txn, values='amount', names='txn_type', title='Transaction Type Share')
fig3.show()

# KPIs
print("📌 Summary Insights")
print(f"Total Transaction Amount: ₹{agg_tx_df['amount'].sum():,.2f}")
print(f"Total Number of Transactions: {agg_tx_df['count'].sum():,}")
print(f"Top Performing Transaction Type: {top_txn.iloc[0]['txn_type']}")

✅ Inserted into SQLite


📌 Summary Insights
Total Transaction Amount: ₹345,523,780,869,125.62
Total Number of Transactions: 235,284,904,571
Top Performing Transaction Type: Peer-to-peer payments


Save & Download Files

In [26]:
agg_tx_df.to_csv("final_aggregated_data.csv", index=False)
top_txn.to_csv("top_transaction_types.csv", index=False)
trend.to_csv("quarterly_trend.csv", index=False)
# Download all
files.download("final_aggregated_data.csv")
files.download("top_transaction_types.csv")
files.download("quarterly_trend.csv")
files.download("phonepe_data.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>